# UCI Online Retail：Raw 与质量门禁

这个 Notebook 是步骤 7 的可复现审计伴侣。它读取 MySQL 已落库批次，不直接修改数据。

## tl;dr

批次 `7aa2b0f4-12aa-4dfb-890c-ffadc8f9d6fa` 已导入 541,909 行。质量门禁 5 条 BLOCKER 全部通过；6 条 WARNING 已记录。Staging 共有 524,878 条有效销售行。下方输出保留这些结论的数据库审计证据。

## Context & Methods

### Key Assumptions

- 输入文件为 `data/inbound/uci_online_retail/Online Retail.xlsx`。
- `B` 类规则是阻断规则；`W` 类规则记录业务可用性风险但不阻断进入 Staging。
- Staging 以 `is_valid_sale_line` 供后续销售指标使用，不删除原始行。

## Data

来源：UCI Machine Learning Repository 的 Online Retail 数据集。数据库连接参数从项目根目录 `.env` 读取。

In [1]:
from pathlib import Path
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(project_root / '.env')
batch_id = '7aa2b0f4-12aa-4dfb-890c-ffadc8f9d6fa'

connection = pymysql.connect(
    host=os.environ.get('MYSQL_HOST', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USER', 'root'),
    password=os.environ['MYSQL_ROOT_PASSWORD'],
    charset='utf8mb4',
)

In [2]:
batch_audit = pd.read_sql(
    '''
    SELECT batch_id, source_file, source_sha256, extract_start_date, extract_end_date,
           rows_loaded, status, started_at, finished_at
    FROM insightops_ops.ingestion_batch
    WHERE batch_id = %s
    ''',
    connection,
    params=(batch_id,),
)
batch_audit

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_2697/2404278727.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  batch_audit = pd.read_sql(


,batch_id,source_file,source_sha256,extract_start_date,extract_end_date,rows_loaded,status,started_at,finished_at
0,7aa2b0f4-12aa-4dfb-890c-ffadc8f9d6fa,Online Retail.xlsx,43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93...,2010-12-01,2011-12-09,541909,SUCCEEDED,2026-08-23 12:19:02.679415,2026-08-23 04:19:17.180967


## Results

In [3]:
quality_results = pd.read_sql(
    '''
    SELECT result.rule_id, result.severity, result.failed_rows, result.failed_rate,
           result.outcome, quality.status AS quality_status
    FROM insightops_ops.data_quality_result AS result
    JOIN insightops_ops.data_quality_run AS quality ON quality.run_id = result.run_id
    WHERE quality.batch_id = %s
    ORDER BY result.severity, result.rule_id
    ''',
    connection,
    params=(batch_id,),
)
quality_results

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_2697/2991505879.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  quality_results = pd.read_sql(


,rule_id,severity,failed_rows,failed_rate,outcome,quality_status
0,B01_batch_succeeded,BLOCKER,0,0.000000,PASSED,SUCCEEDED
1,B02_rows_present,BLOCKER,0,0.000000,PASSED,SUCCEEDED
2,B03_source_row_number_unique,BLOCKER,0,0.000000,PASSED,SUCCEEDED
3,B04_date_within_batch_range,BLOCKER,0,0.000000,PASSED,SUCCEEDED
4,B05_core_fields_populated,BLOCKER,0,0.000000,PASSED,SUCCEEDED
5,W01_exact_duplicate_line,WARNING,5268,0.009721,FAILED,SUCCEEDED
6,W02_cancelled_invoice_line,WARNING,9288,0.017139,FAILED,SUCCEEDED
7,W03_nonpositive_quantity,WARNING,10624,0.019605,FAILED,SUCCEEDED
8,W04_nonpositive_unit_price,WARNING,2517,0.004645,FAILED,SUCCEEDED
9,W05_missing_customer_id,WARNING,135080,0.249267,FAILED,SUCCEEDED


In [4]:
staging_summary = pd.read_sql(
    '''
    SELECT COUNT(*) AS staged_rows,
           SUM(is_valid_sale_line) AS valid_sale_lines,
           SUM(is_cancelled) AS cancelled_lines,
           SUM(is_exact_duplicate) AS duplicate_lines,
           SUM(is_customer_identified) AS identified_customer_lines
    FROM insightops_dw.uci_retail_line_stg
    WHERE batch_id = %s
    ''',
    connection,
    params=(batch_id,),
)
staging_summary

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_2697/2593619665.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  staging_summary = pd.read_sql(


,staged_rows,valid_sale_lines,cancelled_lines,duplicate_lines,identified_customer_lines
0,541909,524878.0,9288.0,5268.0,406829.0


## Takeaways

质量运行状态为 `SUCCEEDED`，因此可以构建订单事实表。GMV 与订单指标只能聚合 `is_valid_sale_line = 1`；客户 RFM/LTV 还必须要求 `is_customer_identified = 1`。完整风险与行数见 `docs/uci_quality_report.md`。